## Acquisition automatique des données brutes

Projet :  Évaluation de la vulnérabilité aux inondations urbaines dans le Grand Casablanca

**Objectif de ce notebook**

Télécharger, extraire et organiser de manière **100 % reproductible** les trois sources de données brutes nécessaires au pipeline :

1. OpenStreetMap (Geofabrik)  
2. Hydrographie CartoMaroc  
3. DEM Grand Casablanca 30 m  

**CRS cible** : `EPSG:26191` (Nord Maroc)  
**Date** : 21 octobre 2026

## Structure des dossiers
```text
donnees_brutes/
├── osm/                                      ← Shapefiles OpenStreetMap (Geofabrik)
├── hydro/
│   ├── hydro_donnees.rar
│   └── extracted/
│       └── hydro-maroc(CartoMaroc)/          ← Données hydrographiques extraites
├── dem/
│   └── DEM_Grand_Casablanca_30m.tif          ← Modèle numérique de terrain (30 m)
├── routes/
│   ├── osm_routes_brut.json                  ← Données brutes Overpass
│   └── routes_casablanca.geojson             ← Réseau viaire au format GeoJSON
├── hcp/                                      ← Données téléchargées depuis le HCP
└── processed/                                ← Données traitées et reprojetées (EPSG:26191)

Installation des bibliothèques

In [1]:
%pip install gdown rarfile tqdm certifi rasterio geopandas -q
print("✓ Packages installés")

Note: you may need to restart the kernel to use updated packages.
✓ Packages installés


## 1. Source OpenStreetMap – Geofabrik

**Source** : https://download.geofabrik.de/africa/morocco-latest-free.shp.zip  
**Format** : Shapefiles thématiques  
**Contenu** : routes, bâtiments, occupation du sol, cours d’eau, POI, etc.

In [13]:
# =============================================================================
# BLOC 1 : TÉLÉCHARGEMENT ET EXTRACTION DES DONNÉES OSM (Geofabrik)
# Objectif : Récupérer automatiquement l'extrait shapefile du Maroc
# Source   : https://download.geofabrik.de
# =============================================================================

from pathlib import Path
import requests
import certifi
import os
from tqdm.auto import tqdm   # barre de progression propre
import zipfile               # nécessaire pour la vérification et l'extraction

# -----------------------------------------------------------------------------
# Correction du problème de certificat SSL (conflit fréquent avec PostgreSQL)
# -----------------------------------------------------------------------------
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

# -----------------------------------------------------------------------------
# Configuration des chemins et de l'URL
# -----------------------------------------------------------------------------
ROOT = Path.cwd().resolve().parent

GEOFABRIK_URL = "https://download.geofabrik.de/africa/morocco-latest-free.shp.zip"

# Dossier de destination situé dans Geodata/donnees_brutes/osm
OSM_DIR = ROOT / "donnees_brutes" / "osm"
OSM_DIR.mkdir(parents=True, exist_ok=True)
# -----------------------------------------------------------------------------
# 1. Nettoyage : suppression d'un éventuel fichier corrompu précédent
# -----------------------------------------------------------------------------
if GEOFABRIK_ZIP.exists():
    print("Suppression du fichier corrompu...")
    GEOFABRIK_ZIP.unlink()

# -----------------------------------------------------------------------------
# 2. Téléchargement robuste avec barre de progression
# -----------------------------------------------------------------------------
print("↓ Nouveau téléchargement Geofabrik...")
response = requests.get(GEOFABRIK_URL, stream=True, verify=certifi.where(), timeout=60)
response.raise_for_status()

total_size = int(response.headers.get("content-length", 0))

with open(GEOFABRIK_ZIP, "wb") as f, tqdm(
    desc="Geofabrik Morocco",
    total=total_size,
    unit="B",
    unit_scale=True,
    unit_divisor=1024,
) as bar:
    for chunk in response.iter_content(chunk_size=1024*1024):  # 1 Mo
        if chunk:
            f.write(chunk)
            bar.update(len(chunk))

print(f"\n✓ Téléchargement terminé : {GEOFABRIK_ZIP.stat().st_size / (1024*1024):.1f} Mo")

# -----------------------------------------------------------------------------
# 3. Vérification de l'intégrité du fichier ZIP
# -----------------------------------------------------------------------------
print("Vérification du zip...")
with zipfile.ZipFile(GEOFABRIK_ZIP, "r") as z:
    print(f"✓ ZIP valide — {len(z.namelist())} fichiers à l’intérieur")
    print("Exemples :", z.namelist()[:6])

# -----------------------------------------------------------------------------
# 4. Extraction des shapefiles dans le dossier osm/
# -----------------------------------------------------------------------------
print("\n→ Extraction en cours (peut prendre 1-2 minutes)...")
with zipfile.ZipFile(GEOFABRIK_ZIP, "r") as z:
    z.extractall(OSM_DIR)
print("✓ Extraction terminée")



Suppression du fichier corrompu...
↓ Nouveau téléchargement Geofabrik...


Geofabrik Morocco:   0%|          | 0.00/470M [00:00<?, ?B/s]


✓ Téléchargement terminé : 469.8 Mo
Vérification du zip...
✓ ZIP valide — 101 fichiers à l’intérieur
Exemples : ['README', 'gis_osm_adminareas_a_free_1.cpg', 'gis_osm_adminareas_a_free_1.dbf', 'gis_osm_adminareas_a_free_1.prj', 'gis_osm_adminareas_a_free_1.shp', 'gis_osm_adminareas_a_free_1.shx']

→ Extraction en cours (peut prendre 1-2 minutes)...
✓ Extraction terminée


## 2. Source Hydrographie – CartoMaroc

**Source** : Google Drive (fichier .rar)  
**Contenu** :  
- Oueds (normal + détaillé)  
- Lacs et Dayas  
- Barrages  
- Bassins versants  
- Nappes phréatiques  

Ces couches sont essentielles pour calculer la distance aux cours d’eau et identifier les zones d’accumulation.

In [14]:

# =============================================================================
# BLOC 2 : TÉLÉCHARGEMENT ET EXTRACTION DES DONNÉES HYDROGRAPHIQUES
# Source   : CartoMaroc (barrages, oueds, lacs, nappes, bassins...)
# Format   : .rar hébergé sur Google Drive
# =============================================================================

from pathlib import Path
import gdown
import subprocess
import shutil
from datetime import datetime

# -----------------------------------------------------------------------------
# Configuration des chemins
# -----------------------------------------------------------------------------

# Le notebook est exécuté depuis Geodata/scripts
ROOT = Path.cwd().resolve().parent

# Dossier hydrographique dans Geodata/donnees_brutes/hydro
hydro_DIR = ROOT / "donnees_brutes" / "hydro"
hydro_DIR.mkdir(parents=True, exist_ok=True)

FILE_ID = "1w0md7nxYZzoyOhy364yMrdVsUiDpaJ-W"

RAR_PATH = hydro_DIR / "hydro_donnees.rar"
EXTRACT_DIR = hydro_DIR / "extracted"

print("Dossier racine :", ROOT)
print("Dossier hydrographique :", hydro_DIR.resolve())
print("Archive RAR :", RAR_PATH.resolve())
print("Dossier d'extraction :", EXTRACT_DIR.resolve())

# -----------------------------------------------------------------------------
# 1. Téléchargement automatique depuis Google Drive
# -----------------------------------------------------------------------------
if RAR_PATH.exists():
    print(f"✓ Fichier .rar déjà présent ({RAR_PATH.stat().st_size / (1024*1024):.1f} Mo)")
else:
    print("↓ Téléchargement depuis Google Drive...")
    gdown.download(id=FILE_ID, output=str(RAR_PATH), quiet=False)
    print(f"✓ Téléchargé → {RAR_PATH}")

# -----------------------------------------------------------------------------
# 2. Extraction avec WinRAR (méthode la plus fiable sur Windows)
# -----------------------------------------------------------------------------
# Nettoyage du dossier d'extraction précédent
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Recherche de WinRAR
possible_winrar = [
    r"C:\Program Files\WinRAR\WinRAR.exe",
    r"C:\Program Files (x86)\WinRAR\WinRAR.exe",
]
winrar = next((p for p in possible_winrar if Path(p).exists()), None)

if winrar is None:
    print("❌ WinRAR non trouvé. Installez-le ou extrayez manuellement une seule fois.")
else:
    print(f"✓ WinRAR trouvé : {winrar}")
    print("→ Extraction en cours...")
    
    cmd = [winrar, "x", "-y", str(RAR_PATH), str(EXTRACT_DIR) + "\\"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✓ Extraction réussie avec WinRAR")
    else:
        print("Erreur lors de l'extraction :", result.stderr or result.stdout)

# -----------------------------------------------------------------------------
# 3. Vérification rapide des fichiers extraits
# -----------------------------------------------------------------------------
print("\nFichiers hydrographiques extraits :")
hydro_dir = EXTRACT_DIR / "hydro-maroc(CartoMaroc)"
if hydro_dir.exists():
    for f in sorted(hydro_dir.glob("*.shp")):
        print(f"  - {f.name} ({f.stat().st_size / 1024:.0f} Ko)")
else:
    print("  ⚠ Dossier hydro-maroc(CartoMaroc) non trouvé")

Dossier racine : C:\Users\espacegamers.ma\Downloads\Geodata
Dossier hydrographique : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\hydro
Archive RAR : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\hydro\hydro_donnees.rar
Dossier d'extraction : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\hydro\extracted
↓ Téléchargement depuis Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1w0md7nxYZzoyOhy364yMrdVsUiDpaJ-W
From (redirected): https://drive.google.com/uc?id=1w0md7nxYZzoyOhy364yMrdVsUiDpaJ-W&confirm=t&uuid=3d6c4329-fe10-4a44-91fc-a6b67ad841f0
To: C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\hydro\hydro_donnees.rar
100%|██████████| 67.1M/67.1M [00:01<00:00, 34.8MB/s]


✓ Téléchargé → C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\hydro\hydro_donnees.rar
✓ WinRAR trouvé : C:\Program Files\WinRAR\WinRAR.exe
→ Extraction en cours...
✓ Extraction réussie avec WinRAR

Fichiers hydrographiques extraits :
  - barrages_(CartoMaroc).shp (4 Ko)
  - Bassin-maroc(CartoMaroc).shp (786 Ko)
  - Lac+Dayat(CartoMaroc).shp (2666 Ko)
  - Nappes_(CartoMaroc).shp (46 Ko)
  - Oued(CartoMaroc) (1).shp (44725 Ko)
  - Oued-Détaillé(CartoMaroc).shp (99980 Ko)


## 3. Source DEM – Grand Casablanca 30 m

**Source** : Google Drive  
**Résolution** : 30 mètres 
**Objectif** : Ce script télécharge le MNT Copernicus 30m
**Utilité** : altitude, pente, direction d’écoulement, Topographic Wetness Index (TWI)

In [15]:
# =============================================================================
# BLOC 3 : TÉLÉCHARGEMENT DU DEM GRAND CASABLANCA 30 m
# Source   : Google Drive
# Résolution : 30 mètres
# =============================================================================

from pathlib import Path
import gdown
from datetime import datetime
import rasterio


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Le notebook est exécuté depuis Geodata/scripts
ROOT = Path.cwd().resolve().parent

# Dossier du DEM dans Geodata/donnees_brutes/dem
DEM_DIR = ROOT / "donnees_brutes" / "dem"
DEM_DIR.mkdir(parents=True, exist_ok=True)

FILE_ID = "15qpvrKivrA5oO6AtYLPgZVen9tUmij6w"
DEM_PATH = DEM_DIR / "DEM_Grand_Casablanca_30m.tif"

print("Dossier racine :", ROOT)
print("Dossier DEM :", DEM_DIR.resolve())
print("Fichier DEM :", DEM_PATH.resolve())

# -----------------------------------------------------------------------------
# 1. Téléchargement
# -----------------------------------------------------------------------------
if DEM_PATH.exists():
    print(f"✓ DEM déjà présent : {DEM_PATH.name}")
    print(f"  Taille : {DEM_PATH.stat().st_size / (1024*1024):.1f} Mo")
else:
    print("↓ Téléchargement du DEM depuis Google Drive...")
    gdown.download(id=FILE_ID, output=str(DEM_PATH), quiet=False)
    print(f"✓ Téléchargé → {DEM_PATH}")

# -----------------------------------------------------------------------------
# 2. Vérification de l'intégrité du raster
# -----------------------------------------------------------------------------
print("\n→ Vérification du raster...")
try:
    with rasterio.open(DEM_PATH) as src:
        print(f"  CRS              : {src.crs}")
        print(f"  Dimensions       : {src.width} × {src.height}")
        print(f"  Résolution       : {src.res}")
        print(f"  Nombre de bandes : {src.count}")
        print(f"  Type de données  : {src.dtypes[0]}")
        print(f"  Emprise          : {src.bounds}")
    print("✓ DEM valide et lisible")
except Exception as e:
    print(f"❌ Erreur de lecture du DEM : {e}")

Dossier racine : C:\Users\espacegamers.ma\Downloads\Geodata
Dossier DEM : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\dem
Fichier DEM : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\dem\DEM_Grand_Casablanca_30m.tif
↓ Téléchargement du DEM depuis Google Drive...


Downloading...
From: https://drive.google.com/uc?id=15qpvrKivrA5oO6AtYLPgZVen9tUmij6w
To: C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\dem\DEM_Grand_Casablanca_30m.tif
100%|██████████| 18.0M/18.0M [00:00<00:00, 50.9MB/s]


✓ Téléchargé → C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\dem\DEM_Grand_Casablanca_30m.tif

→ Vérification du raster...
  CRS              : GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
  Dimensions       : 8494 × 5682
  Résolution       : (0.0002694945852358564, 0.0002694945852358564)
  Nombre de bandes : 1
  Type de données  : int16
  Emprise          : BoundingBox(left=-9.051369141642851, bottom=32.300946429177095, right=-6.762282134649487, top=33.83221466248723)
✓ DEM valide et lisible


## 4.RÉSEAU VIAIRE & INFRASTRUCTURES DE MOBILITÉ (ROUTES, TRAMWAY)

In [12]:
# -----------------------------------------------------------------------------
# SOURCE 2 : RÉSEAU VIAIRE — CASABLANCA (OpenStreetMap / Overpass)
# -----------------------------------------------------------------------------

from pathlib import Path
import json
import urllib.request
import urllib.parse
import urllib.error

# 1. CONFIGURATION DES DOSSIERS
# Dossier parent de scripts : Geodata
ROOT = Path.cwd().resolve().parent

# Dossier de sortie situé à l'extérieur de scripts
routes_DIR = ROOT / "donnees_brutes" / "routes"
routes_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier racine :", ROOT)
print("Dossier de sortie :", routes_DIR.resolve())


# Emprise géographique : Casablanca et ses environs
# Format Overpass : sud, ouest, nord, est
BBOX = "33.45,-7.80,33.75,-7.35"

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

HEADERS = {
    "User-Agent": "GeodataCasablancaInondations/1.0",
    "Accept": "application/json"
}


# 2. AFFICHAGE DES ÉTAPES
def log_etape(message):
    print("\n" + "=" * 75)
    print(message)
    print("=" * 75)


# 3. ACQUISITION DU RÉSEAU VIAIRE
def acquerir_routes():

    log_etape(" EXTRACTION DU RÉSEAU VIAIRE")

    query = f"""
    [out:json][timeout:60];
    (
      way["highway"~"^(motorway|trunk|primary|secondary)$"]({BBOX});
    );
    out body;
    >;
    out skel qt;
    """

    req = urllib.request.Request(
        OVERPASS_URL,
        data=urllib.parse.urlencode({"data": query}).encode("utf-8"),
        headers={
            **HEADERS,
            "Content-Type": "application/x-www-form-urlencoded"
        },
        method="POST"
    )

    print("[+] Interrogation du serveur Overpass...")

    try:
        with urllib.request.urlopen(req, timeout=90) as resp:
            res = json.loads(resp.read().decode("utf-8"))

    except urllib.error.HTTPError as e:
        print(f"[ERREUR HTTP] {e.code} : {e.reason}")
        return None

    except (urllib.error.URLError, TimeoutError,
            json.JSONDecodeError, OSError) as e:
        print(f"[ERREUR] Échec de la requête Overpass : {e}")
        return None

    # Sauvegarde des données brutes
    raw_path = routes_DIR / "osm_routes_brut.json"

    with open(raw_path, "w", encoding="utf-8") as f:
        json.dump(res, f, ensure_ascii=False, indent=2)

    elements = res.get("elements", [])

    # Index des nœuds OSM
    nodes = {
        el["id"]: (el["lon"], el["lat"])
        for el in elements
        if el.get("type") == "node"
        and "lon" in el and "lat" in el
    }

    features = []

    # Reconstruction des géométries routières
    for el in elements:

        if el.get("type") != "way":
            continue

        coords = [
            nodes[node_id]
            for node_id in el.get("nodes", [])
            if node_id in nodes
        ]

        if len(coords) < 2:
            continue

        tags = el.get("tags", {})

        features.append({
            "type": "Feature",
            "properties": {
                "osm_id": el["id"],
                "nom_rue": tags.get("name", "Axe non nommé"),
                "type_voie": tags.get("highway"),
                "surface": tags.get("surface"),
                "lanes": tags.get("lanes"),
                "bridge": tags.get("bridge"),
                "tunnel": tags.get("tunnel")
            },
            "geometry": {
                "type": "LineString",
                "coordinates": coords
            }
        })

    # Export GeoJSON
    geojson_path = routes_DIR / "routes_casablanca.geojson"

    geojson = {
        "type": "FeatureCollection",
        "name": "routes_casablanca",
        "features": features
    }

    with open(geojson_path, "w", encoding="utf-8") as f:
        json.dump(geojson, f, ensure_ascii=False, indent=2)

    print(f"\n[OK] {len(features)} axes routiers enregistrés.")
    print(f"[OK] JSON brut : {raw_path.resolve()}")
    print(f"[OK] GeoJSON   : {geojson_path.resolve()}")

    if not features:
        print("[ATTENTION] Aucun axe récupéré.")

    return str(geojson_path)


# 4. EXÉCUTION
routes_path = acquerir_routes()

if routes_path:
    print("\n✓ Acquisition terminée.")
else:
    print("\n⚠ Acquisition non terminée.")

Dossier racine : C:\Users\espacegamers.ma\Downloads\Geodata
Dossier de sortie : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\routes

 EXTRACTION DU RÉSEAU VIAIRE
[+] Interrogation du serveur Overpass...

[OK] 4285 axes routiers enregistrés.
[OK] JSON brut : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\routes\osm_routes_brut.json
[OK] GeoJSON   : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\routes\routes_casablanca.geojson

✓ Acquisition terminée.


## 5.DONNÉES STATISTIQUES DU HCP

In [16]:
# -----------------------------------------------------------------------------
# SOURCE HCP : TÉLÉCHARGEMENT ET STOCKAGE DES DONNÉES BRUTES
# -----------------------------------------------------------------------------

from pathlib import Path
import urllib.request
import urllib.error
import mimetypes


# 1. Configuration
# Le notebook est exécuté depuis Geodata/scripts
ROOT = Path.cwd().resolve().parent

# Dossier de stockage HCP dans Geodata/donnees_brutes/hcp
hcp_DIR = ROOT / "donnees_brutes" / "hcp"
hcp_DIR.mkdir(parents=True, exist_ok=True)

URL_HCP = "https://www.hcp.ma/file/242341/"

print("Dossier racine :", ROOT)
print("Dossier HCP :", hcp_DIR.resolve())


# 2. Téléchargement
req = urllib.request.Request(
    URL_HCP,
    headers={
        "User-Agent": "Mozilla/5.0",
        "Accept": "*/*"
    }
)

try:
    with urllib.request.urlopen(req, timeout=120) as response:
        content = response.read()

        content_type = response.headers.get(
            "Content-Type", "application/octet-stream"
        )
        disposition = response.headers.get(
            "Content-Disposition", ""
        )
        final_url = response.geturl()

    if not content:
        raise ValueError("La réponse du serveur est vide.")

    # 3. Détection du nom et de l'extension
    import re

    match = re.search(
        r'filename\*?=(?:UTF-8\'\')?["\']?([^"\';]+)',
        disposition,
        flags=re.IGNORECASE
    )

    if match:
        nom_fichier = match.group(1).strip()
    else:
        extension = mimetypes.guess_extension(
            content_type.split(";")[0].strip()
        )

        # Détection par signature si le serveur ne précise pas le format
        if content.startswith(b"PK\x03\x04"):
            extension = ".zip"
        elif content.startswith(b"\xD0\xCF\x11\xE0"):
            extension = ".xls"
        elif content.startswith(b"%PDF-"):
            extension = ".pdf"
        elif content.lstrip().startswith((b"{", b"[")):
            extension = ".json"
        elif content.lstrip().startswith(b"<?xml"):
            extension = ".xml"

        if not extension or extension == ".bin":
            extension = ".bin"

        nom_fichier = "hcp_242341" + extension

    # Sécuriser le nom du fichier
    nom_fichier = Path(nom_fichier).name
    fichier_path = hcp_DIR / nom_fichier

    # Éviter d'enregistrer une page HTML d'erreur comme donnée brute
    debut = content[:1000].lstrip().lower()

    if (
        "text/html" in content_type.lower()
        and (b"<html" in debut or b"<!doctype html" in debut)
    ):
        raise ValueError(
            "Le serveur a renvoyé une page HTML au lieu du fichier. "
            "Vérifie que l'URL correspond bien à une ressource téléchargeable."
        )

    with open(fichier_path, "wb") as f:
        f.write(content)

    print("\n[OK] Téléchargement terminé.")
    print("Fichier :", fichier_path.resolve())
    print("Type HTTP :", content_type)
   
    print(f"Taille : {len(content) / (1024 * 1024):.2f} Mo")

except urllib.error.HTTPError as e:
    print(f"[ERREUR HTTP] {e.code} : {e.reason}")

except (urllib.error.URLError, TimeoutError, OSError, ValueError) as e:
    print(f"[ERREUR] {e}")

Dossier racine : C:\Users\espacegamers.ma\Downloads\Geodata
Dossier HCP : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\hcp

[OK] Téléchargement terminé.
Fichier : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\hcp\Population lÃ©gale du Royaume du Maroc selon les rÃ©sultats du RGPH 2024, format EXCEL.xlsx
Type HTTP : application/octet-stream
Taille : 0.16 Mo


## 6. Vérification globale des données brutes

Cette section contrôle que toutes les sources sont bien présentes et lisibles.

In [17]:

from pathlib import Path
import json

print("=" * 70)
print("RÉSUMÉ DES DONNÉES BRUTES TÉLÉCHARGÉES")
print("=" * 70)

# 1. OSM — GEOFABRIK
osm_dir = Path("donnees_brutes/osm")

print("\n1. OpenStreetMap (Geofabrik)")
print(f"   Dossier : {osm_dir}")
print(f"   Fichiers .shp trouvés : {len(list(osm_dir.rglob('*.shp')))}")

# 2. HYDROGRAPHIE — CARTOMAROC
hydro_dir = Path(
    "donnees_brutes/source2/extracted/hydro-maroc(CartoMaroc)"
)

print("\n2. Hydrographie CartoMaroc")
print(f"   Dossier : {hydro_dir}")

if hydro_dir.exists():
    print(
        f"   Fichiers .shp trouvés : "
        f"{len(list(hydro_dir.rglob('*.shp')))}"
    )
else:
    print("   ⚠ Dossier non trouvé")

# 3. MODÈLE NUMÉRIQUE DE TERRAIN (DEM)
dem_path = Path("donnees_brutes/dem/DEM_Grand_Casablanca_30m.tif")

print("\n3. DEM Grand Casablanca 30 m")
print(f"   Fichier : {dem_path}")
print(f"   Présent : {dem_path.exists()}")

if dem_path.exists():
    print(f"   Taille : {dem_path.stat().st_size / (1024 * 1024):.1f} Mo")

# 4. RÉSEAU VIAIRE — OVERPASS / OPENSTREETMAP
routes_dir = Path("donnees_brutes/routes")

print("\n4. Réseau viaire (OpenStreetMap / Overpass)")
print(f"   Dossier : {routes_dir}")

if routes_dir.exists():
    raw_path = routes_dir / "osm_routes_brut.json"
    geojson_path = routes_dir / "routes_casablanca.geojson"

    print(f"   JSON brut présent : {raw_path.exists()}")
    print(f"   GeoJSON présent   : {geojson_path.exists()}")

    if raw_path.exists():
        print(
            f"   Taille JSON brut : "
            f"{raw_path.stat().st_size / (1024 * 1024):.2f} Mo"
        )

    if geojson_path.exists():
        print(
            f"   Taille GeoJSON   : "
            f"{geojson_path.stat().st_size / (1024 * 1024):.2f} Mo"
        )

        try:
            with open(geojson_path, "r", encoding="utf-8") as f:
                routes_data = json.load(f)

            nb_routes = len(routes_data.get("features", []))
            print(f"   Axes routiers enregistrés : {nb_routes}")

        except (json.JSONDecodeError, OSError) as e:
            print(f"   ⚠ Lecture du GeoJSON impossible : {e}")
else:
    print("   ⚠ Dossier non trouvé")

# 5. DONNÉES STATISTIQUES — HCP
hcp_dir = Path("donnees_brutes/hcp")

print("\n5. Données statistiques du HCP")
print(f"   Dossier : {hcp_dir}")

if hcp_dir.exists():
    fichiers_hcp = [f for f in hcp_dir.rglob("*") if f.is_file()]

    print(f"   Nombre de fichiers : {len(fichiers_hcp)}")

    if fichiers_hcp:
        for fichier in fichiers_hcp:
            taille_mo = fichier.stat().st_size / (1024 * 1024)
            print(f"   - {fichier.name} ({taille_mo:.2f} Mo)")
    else:
        print("   ⚠ Aucun fichier téléchargé")
else:
    print("   ⚠ Dossier non trouvé")

# BILAN FINAL
print("\n" + "=" * 70)
print("✓ Vérification terminée")
print("=" * 70)

RÉSUMÉ DES DONNÉES BRUTES TÉLÉCHARGÉES

1. OpenStreetMap (Geofabrik)
   Dossier : donnees_brutes\osm
   Fichiers .shp trouvés : 20

2. Hydrographie CartoMaroc
   Dossier : donnees_brutes\source2\extracted\hydro-maroc(CartoMaroc)
   ⚠ Dossier non trouvé

3. DEM Grand Casablanca 30 m
   Fichier : donnees_brutes\dem\DEM_Grand_Casablanca_30m.tif
   Présent : True
   Taille : 17.2 Mo

4. Réseau viaire (OpenStreetMap / Overpass)
   Dossier : donnees_brutes\routes
   JSON brut présent : True
   GeoJSON présent   : True
   Taille JSON brut : 6.30 Mo
   Taille GeoJSON   : 4.57 Mo
   Axes routiers enregistrés : 4285

5. Données statistiques du HCP
   Dossier : donnees_brutes\hcp
   Nombre de fichiers : 1
   - Population lÃ©gale du Royaume du Maroc selon les rÃ©sultats du RGPH 2024, format EXCEL.xlsx (0.16 Mo)

✓ Vérification terminée


## Fin de l’acquisition des données brutes

Les données brutes sont conservées séparément afin de préserver les sources originales et de garantir la reproductibilité du workflow.

**Prochaine étape** : 
→ Découpage spatial des données
- Utilisation de la limite administrative de la région Casablanca-Settat comme emprise d’étude.
- Découpage de toutes les couches pertinentes de la première source : OpenStreetMap (OSM).
- Découpage de toutes les couches pertinentes de la deuxième source : données hydrographiques CartoMaroc.
- Enregistrement des couches découpées dans donnees_traitees/casablanca_settat/, en conservant une organisation distincte pour chaque source.
  
Les données découpées seront ensuite préparées pour les étapes suivantes du projet : reprojection vers EPSG:26191, harmonisation des couches et intégration dans PostGIS.